# DIL Results Analysis
Use the helper functions in `data_analysis_fcns.DIL_Metrics` to visualize confusion matrices, imbalanced-learning metrics, and population statistics across multiple runs.

In [1]:
# Imports and helper fcns
from data_analysis_fcns.DIL_Metrics import (
    plot_confusion_matrices,
    plot_imbalanced_metrics,
    plot_population_statistics,
)

from pathlib import Path
import json
import re

def read_jsonc(path: str):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    raw = re.sub(r"//.*", "", raw)
    raw = re.sub(r"/\*.*?\*/", "", raw, flags=re.DOTALL)
    return json.loads(raw)


In [2]:
# --- Configure which configs to compare (use config names without .jsonc)

metaconfig = "meta_configs/core50_DIL_retry.jsonc" # If metaconfig is used, configs is ignored
configs = []
metapath = Path(metaconfig)
if metapath.is_file():
    metadata = read_jsonc(metaconfig)
    configs = metadata.get("configs")
    configs = [configname.replace('configs/', '') for configname in configs]
    configs = [configname.replace('.jsonc', '') for configname in configs]
    
else:
    configs = ["mini_debug", "example_config"]  # adjust to your setup

labels = None  # optional display labels for the configs
# Example single result file for quick inspection
result_file = 'results/mini_debug_vit_imagelevel/vit_moe_imagelevel_cifar10_04241501.pt'  # change to one of your saved .pt files

## Confusion matrices
Plot per-domain confusion matrices (model vs baseline) and/or epoch-by-epoch full-test confusion matrices. If you leave all options unset the function will plot all available confusion data.

In [ ]:
# Plot domain-level comparisons and the epoch list (toggle arguments as desired)
plot_confusion_matrices(result_file, per_domain=True, compare_baseline=True, per_epoch=False)

## Imbalanced-learning metrics per epoch
Compute macro-averaged recall/precision/F1 plus MAUC and G-Mean per epoch from the epoch-level confusion matrices.

In [ ]:
# Plot imbalanced-learning metrics for the chosen result file
plot_imbalanced_metrics(result_file)

## Population statistics across configs
Aggregate multiple runs per config (searching under the `results/` tree for filenames or folders that contain each config name) and plot mean +/- std shading for each metric.

In [ ]:
# Compare configured experiments using saved .pt files under results/
plot_population_statistics(configs, results_root='results', labels=labels)

## Expert usage analysis
Plot per-layer expert utilization stored in the results file (if available).

In [3]:
# Load saved results and plot expert usage per layer
import torch
from pathlib import Path
import matplotlib.pyplot as plt

def plot_expert_usage(result_path):
    data = torch.load(result_path)
    cum = data.get('expert_cumulative_usage', None)
    per_epoch = data.get('expert_usage_history', None)
    if cum is None and per_epoch is None:
        print('No expert usage data found in', result_path)
        return
    # plot cumulative usage if present (list of lists per layer)
    if cum is not None:
        for li, layer in enumerate(cum):
            plt.figure()
            plt.bar(range(len(layer)), layer)
            plt.title(f'Layer {li} cumulative image counts per expert')
            plt.xlabel('expert')
            plt.ylabel('image count')
            plt.show()
    # plot last epoch per-layer usage (if history exists)
    if per_epoch is not None and len(per_epoch) > 0:
        last = per_epoch[-1]
        # last expected to be a list of per-layer lists (or None)
        for li, layer in enumerate(last):
            try:
                plt.figure()
                plt.bar(range(len(layer)), layer)
                plt.title(f'Layer {li} epoch image counts per expert (last)')
                plt.xlabel('expert')
                plt.ylabel('image count')
                plt.show()
            except Exception:
                pass

# call with the chosen result file
plot_expert_usage(result_file)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

### Expert usage histograms after each domain
The following cell loads the saved results file and plots per-layer expert usage histograms taken at the epoch where each domain finished.
If confusion matrices are available in the saved file, an approximate per-class per-expert heatmap is also shown (approximation distributes expert counts proportionally to class frequencies).

In [ ]:
# Plot per-domain expert usage histograms (taken at domain boundaries)
import torch
import numpy as np
import matplotlib.pyplot as plt

data = torch.load(result_file, weights_only=False)
expert_hist = data.get('expert_usage_history', None)
domain_boundaries = data.get('domain_boundaries', None)
confusions = data.get('confusion_matrices', None)

if expert_hist is None or domain_boundaries is None:
    print('No expert usage or domain boundary information found in', result_file)
else:
    num_domains = len(domain_boundaries)
    for d in range(num_domains):
        b = domain_boundaries[d]
        epoch_idx = max(0, b - 1)
        if epoch_idx >= len(expert_hist) or expert_hist[epoch_idx] is None:
            print(f'No expert-usage snapshot available for domain {d} at epoch {epoch_idx}')
            continue
        snapshot = expert_hist[epoch_idx]  # list over domains (one entry per domain)
        if d >= len(snapshot) or snapshot[d] is None:
            print(f'No domain usage entry for domain {d} in snapshot at epoch {epoch_idx}')
            continue
        domain_usage = snapshot[d]  # expected: list of per-layer lists (counts per expert)
        # normalize structure and find sizes
        layers = [np.array(l, dtype=float) if l is not None else np.array([]) for l in domain_usage]
        num_layers = len(layers)
        num_experts = int(layers[0].size) if num_layers > 0 and layers[0].size > 0 else 0
        if num_layers == 0 or num_experts == 0:
            print(f'Domain {d}: no expert counts available')
            continue
        # plot bar for each layer
        fig, axes = plt.subplots(1, num_layers, figsize=(4 * max(1, num_layers), 4))
        if num_layers == 1:
            axes = [axes]
        for li in range(num_layers):
            ax = axes[li]
            counts = layers[li] if layers[li].size > 0 else np.zeros(num_experts)
            ax.bar(np.arange(len(counts)), counts, color='C0')
            ax.set_title(f'Domain {d} - Layer {li} expert counts (epoch {epoch_idx})')
            ax.set_xlabel('expert')
            ax.set_ylabel('image count')
        plt.tight_layout()
        plt.show()

        # Aggregate across layers and attempt approximate per-class breakdown if confusion matrix available
        agg = np.sum(np.vstack([layers[l] if layers[l].size > 0 else np.zeros(num_experts) for l in range(num_layers)]), axis=0)
        if confusions is None:
            print('No confusion matrices found in saved results; cannot compute per-class expert breakdown exactly.')
            continue
        if d >= len(confusions) or confusions[d] is None:
            print(f'No confusion matrix for domain {d} — skipping per-class approx')
            continue
        cm = np.array(confusions[d], dtype=float)
        class_counts = cm.sum(axis=1)
        if class_counts.sum() <= 0:
            print(f'No class samples for domain {d} — skipping per-class approx')
            continue
        # approximate: distribute aggregate expert counts proportionally to class frequencies (note: approximate only)
        approx = np.outer(class_counts / class_counts.sum(), agg)
        fig, ax = plt.subplots(figsize=(6, max(4, 0.2 * approx.shape[0])))
        im = ax.imshow(approx, aspect='auto', cmap='viridis')
        ax.set_xlabel('expert')
        ax.set_ylabel('class')
        ax.set_title(f'Approx. expert usage per class (Domain {d}) — distributed proportionally to class counts')
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label('approx image count')
        plt.show()

# End

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 